In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=['course'],
    mode='ivf',
    db_path='faq_vectors2.db'
)

In [3]:
query = 'I just discovered the course. Can I still join it?'
query_vector = model.encode(query)

results = vs_index.search(
    query_vector,
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=5
)

In [4]:
results

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.',
  'doc_id': '69d122f12e'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offere

In [5]:
from rag_helper import RAGBase

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {'course': self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [6]:
from dotenv import load_dotenv
from openai import OpenAI
import os
load_dotenv()

# uncomment below to use openai
# openai_client = OpenAI()

# uncomment below if using groq instead
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [7]:
vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=openai_client,
    model='qwen/qwen3-32b')   # if using openai instead of groq, no need to pass model parameter

In [8]:
vector_assistant.rag('the program has already begun, can I still sign up?')

'Yes, you can still join the course even if it has already begun. Here’s what you need to know:\n\n1. **Late Enrollment**: You’re welcome to start learning and submitting homework immediately, as registration is only used to gauge interest. There’s no need to wait for a confirmation email or formal registration approval.  \n   \n2. **Certificate Requirement**: To earn a certificate, ensure your project is submitted while the submission window is still open. Submitting late may disqualify you from receiving one.  \n\n3. **Next Session Reminder**: The course will run again in **Summer 2027**, so there’s no urgency to finish immediately if you’re already aware of that timeline.  \n\nFor specifics about homework, use the existing Q&A or Slack for support. Start anytime! 😊'

In [10]:
# should have bee I don't know if using openai(chatgpt) but groq makes up an answer based on its prior info
vector_assistant.rag('whats queen gambit?')

'The **Queen\'s Gambit** is a classic chess opening strategy where a player sacrifices a pawn early in the game (1. d4 d5 2. c4) to gain control of the center and improve piece development. It’s a tactical maneuver requiring careful planning and adaptability to the opponent’s responses.\n\nIn the context of **Agentic RAG**, the analogy is metaphorical:  \nJust as the Queen\'s Gambit involves strategically sacrificing short-term material (a pawn) to gain long-term positional advantage, an agentic RAG system strategically decides *when to use tools/functions* (e.g., searching documents, performing calculations) to optimize responses.  \n\n- **Basic RAG** is like a fixed sequence in chess: retrieval happens linearly before answering.  \n- **Agentic RAG** is more dynamic, like a player adapting mid-game—deciding whether to "make a move" (invoke a tool) based on evolving context.  \n\nThe key idea: **both require foresight and flexibility** to achieve optimal outcomes—whether in chess or mo

In [11]:
vs_index.close()